# 01 — SSM Basics: From Continuous to Discrete

This notebook builds a classical state space model (SSM) from first principles in NumPy.

**What you will learn:**
1. The continuous-time SSM formulation: $x'(t) = Ax(t) + Bu(t)$, $y(t) = Cx(t)$
2. Zero-order hold (ZOH) discretization
3. The discrete recurrence: $x_t = \bar{A}x_{t-1} + \bar{B}u_t$, $y_t = Cx_t$
4. How eigenvalue placement controls stability
5. Multi-dimensional state dynamics


## Continuous-time SSM

A linear time-invariant (LTI) state space model has the form:

$$x'(t) = Ax(t) + Bu(t), \quad y(t) = Cx(t) + Du(t)$$

where:
- $x(t) \in \mathbb{R}^N$ is the hidden state
- $u(t) \in \mathbb{R}^M$ is the input
- $A \in \mathbb{R}^{N \times N}$ controls state dynamics (eigenvalues determine stability)
- $B \in \mathbb{R}^{N \times M}$ maps input to state
- $C \in \mathbb{R}^{P \times N}$ maps state to output

**Key insight:** The eigenvalues of $A$ determine everything about stability. If all eigenvalues have negative real parts, the system is stable.


## Zero-Order Hold (ZOH) Discretization

To run an SSM on a computer, we discretize it with a time step $\Delta$. ZOH assumes the input $u(t)$ is constant over each interval:

$$\bar{A} = e^{A\Delta}, \quad \bar{B} = (e^{A\Delta} - I) A^{-1} B$$

This gives us a discrete recurrence:

$$x_t = \bar{A} x_{t-1} + \bar{B} u_t, \quad y_t = C x_t$$

**Why ZOH matters for Mamba:** In the full Mamba model, $\Delta$ becomes *input-dependent* (selective). But the underlying discretization math is the same ZOH formula.

For a diagonal $A$ (as Mamba uses), ZOH simplifies to:
$$\bar{A}_{ii} = e^{A_{ii} \cdot \Delta}$$

No matrix inverse needed — just an elementwise exp.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def rollout_1d(a_bar: float, b_bar: float, c: float, u: np.ndarray):
    """Run a scalar SSM recurrence: x_t = a_bar * x_{t-1} + b_bar * u_t, y_t = c * x_t."""
    x = 0.0
    xs, ys = [], []
    for u_t in u:
        x = a_bar * x + b_bar * u_t
        y = c * x
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Create a sine wave input
T = 200
t = np.linspace(0, 12, T)
u = np.sin(t)

# Stable vs unstable: |a_bar| < 1 is stable
stable_a, unstable_a = 0.8, 1.02
b, c = 0.2, 1.0

x_stable, y_stable = rollout_1d(stable_a, b, c, u)
x_unstable, y_unstable = rollout_1d(unstable_a, b, c, u)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, x_stable, label=f"stable (a={stable_a})", linewidth=2)
axes[0].plot(t, x_unstable, label=f"unstable (a={unstable_a})", linewidth=2)
axes[0].set_title("Hidden state evolution")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("x_t")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, y_stable, label="stable output", linewidth=2)
axes[1].plot(t, y_unstable, label="unstable output", linewidth=2)
axes[1].set_title("Output")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("y_t")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("1D SSM: Stability depends on |a_bar| < 1", fontsize=13)
fig.tight_layout()
plt.show()

## Stability Sweep

The discrete system is stable when $|\bar{A}| < 1$ (scalar case) or when all eigenvalues of $\bar{A}$ have magnitude $< 1$ (matrix case).

Let us sweep different values to see the transition:


In [ ]:
a_values = [0.5, 0.8, 0.95, 0.99, 1.0, 1.01, 1.05]
u_input = np.sin(np.linspace(0, 8, 150))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for a_val in a_values:
    x, y = rollout_1d(a_val, 0.2, 1.0, u_input)
    style = "-" if a_val < 1.0 else "--"
    axes[0].plot(x, linestyle=style, label=f"a={a_val}", alpha=0.8)
    axes[1].plot(y, linestyle=style, label=f"a={a_val}", alpha=0.8)

for ax, title in zip(axes, ["State trajectories", "Output trajectories"]):
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-3, 3)

fig.suptitle("Stability boundary: |a_bar| = 1", fontsize=13)
fig.tight_layout()
plt.show()

## ZOH Discretization: Block Matrix Exponential

Instead of computing $A^{-1}$ (which can be numerically unstable), we use the block-matrix exponential identity:

$$\exp\left(\Delta \begin{bmatrix} A & B \\ 0 & 0 \end{bmatrix}\right) = \begin{bmatrix} \bar{A} & \bar{B} \\ 0 & I \end{bmatrix}$$

This is the same approach used in `src/mamba_minimal/discretization.py`.


In [ ]:
from scipy.linalg import expm

def zoh_discretize_numpy(A, B, delta):
    """ZOH discretization using the block-matrix exponential trick."""
    N, M = A.shape[0], B.shape[1]
    block = np.zeros((N + M, N + M))
    block[:N, :N] = A
    block[:N, N:] = B
    discrete = expm(delta * block)
    return discrete[:N, :N], discrete[:N, N:]

# 2D system: damped oscillator
A = np.array([[0.0, 1.0], [-1.0, -0.3]])  # oscillator with damping
B = np.array([[0.0], [1.0]])               # force applied to velocity
C = np.array([[1.0, 0.0]])                 # observe position

delta = 0.1
A_bar, B_bar = zoh_discretize_numpy(A, B, delta)

print(f"Continuous A:\n{A}")
print(f"\nDiscrete A_bar (dt={delta}):\n{np.round(A_bar, 5)}")
print(f"\nDiscrete B_bar:\n{np.round(B_bar, 5)}")
eigenvalues = np.linalg.eigvals(A_bar)
print(f"\nEigenvalues of A_bar: {np.round(eigenvalues, 5)}")
print(f"Magnitudes: {np.round(np.abs(eigenvalues), 5)}  (all < 1 => stable)")

## Multi-Dimensional SSM Rollout

In [ ]:
def rollout_matrix(A_bar, B_bar, C, u_sequence):
    """Run x_t = A_bar @ x_{t-1} + B_bar @ u_t, y_t = C @ x_t."""
    N = A_bar.shape[0]
    x = np.zeros(N)
    xs, ys = [], []
    for u_t in u_sequence:
        x = A_bar @ x + B_bar.squeeze() * u_t
        y = (C @ x).item()
        xs.append(x.copy())
        ys.append(y)
    return np.array(xs), np.array(ys)

T = 300
t = np.linspace(0, 10, T)
u_step = np.ones(T); u_step[:T//4] = 0.0
u_chirp = np.sin(t * t * 0.3)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, (u_input, name) in enumerate([(u_step, "Step input"), (u_chirp, "Chirp input")]):
    xs, ys = rollout_matrix(A_bar, B_bar, C, u_input)
    axes[0, col].plot(t, u_input, "k--", alpha=0.5, label="input")
    axes[0, col].plot(t, ys, linewidth=2, label="output y_t")
    axes[0, col].set_title(name)
    axes[0, col].legend()
    axes[0, col].grid(True, alpha=0.3)
    axes[1, col].plot(t, xs[:, 0], label="x[0] (position)")
    axes[1, col].plot(t, xs[:, 1], label="x[1] (velocity)")
    axes[1, col].set_title(f"Hidden state ({name})")
    axes[1, col].legend()
    axes[1, col].grid(True, alpha=0.3)

fig.suptitle("2D SSM: Damped Oscillator (ZOH discretized)", fontsize=13)
fig.tight_layout()
plt.show()

## Connection to Mamba

In the classical SSM above, $A$, $B$, $C$, and $\Delta$ are all **fixed** — the system behaves the same regardless of the input content.

**Mamba changes this fundamentally:**
- $B_t$, $C_t$, and $\Delta_t$ become functions of the current input token
- This makes the system **selective**: it can decide what to remember and what to forget based on content
- The per-token discretization is: $\bar{A}_t = e^{\Delta_t \cdot A}$, $\bar{B}_t = \Delta_t \cdot B_t$

This selectivity is what distinguishes Mamba from earlier SSM architectures like S4 and makes it competitive with Transformers.

**Next notebook:** We implement this selective scan in PyTorch and validate it against the official Mamba model.
